# Forecasting Model Monitoring with Graphana

## Importing

In [1]:
import pandas as pd
import numpy as np
import joblib
import requests
import mlflow
import dagshub
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from datetime import datetime, timedelta
from evidently import Report
from evidently import DataDefinition
from evidently import Dataset
from evidently.metrics import ValueDrift, DriftedColumnsCount, MissingValueCount

## Artifacts Loading

In [2]:
data = joblib.load("../mle-specialization-03/data/processed/processed_data.joblib")
X_train = data["X_train"]
y_train = data["y_train"]
X_test = data["X_test"]
y_test = data["y_test"]

In [3]:
y_pred = pd.read_csv("../mle-specialization-03/data/processed/Ridge Stacking Ensemble_predictions.csv")
y_pred

,Close
0,141.801469
1,143.431635
2,146.811457
3,145.692230
4,147.349646
...,...
120784,61.919983
120785,62.829972
120786,62.501406
120787,61.634262


In [4]:
"""
Loading the PROD model from MlFlow (DagsHub)
"""
mlflow.set_tracking_uri("https://dagshub.com/MaCh1Ne01/mle-specialization-03.mlflow")
run_id = "3264ea81c3c54f38976b72b6c39ca278"
model_name = "Ridge Stacking Ensemble"
best_model = mlflow.sklearn.load_model(f"runs:/{run_id}/{model_name}")

/home/marcos/grafana-ml-monitoring/graph-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Loading to InfluxDB

In [5]:
db_params = {
    "db": "evidently_metrics",
    "u": "admin",
    "p": "admin",
    "precision": "ns"
}

headers = {
    "Content-Type": "text/plain; charset=utf-8",
}

BASE_URL = "http://localhost:8086"
DATA_POINTS = 100
r2_score_data = []

for i in range(DATA_POINTS):
    timestamp = datetime.now() - timedelta(hours=100) + timedelta(hours=i)
    simulated_r2_score =  round(r2_score(y_test, y_pred) + np.random.normal(0.03, 0.01), 4)
    line = f"forecasting_model_performance r2_score={simulated_r2_score} {int(timestamp.timestamp()) * 1000000000}"
    r2_score_data.append(line)

payload='\n'.join(r2_score_data)
write_r = requests.post(f"{BASE_URL}/write", params=db_params,  data=payload, headers=headers)

## Drift Report with Evidently

In [6]:
report = Report(metrics = [
    DriftedColumnsCount(method="psi")
])

drift_report = report.run(reference_data=X_train, current_data=X_test)

In [7]:
drifted_columns_count = int(drift_report.dict()["metrics"][0]["value"]["count"])
for i in range(100):
    timestamp = datetime.now() - timedelta(hours=100) + timedelta(hours=i)
    drift_payload = f"forecasting_model_drift_metrics drifted_columns_count={drifted_columns_count + np.random.randint(10)} {int(timestamp.timestamp()) * 1000000000}"
    write_drift = requests.post(f"{BASE_URL}/write", params=db_params,  data=drift_payload, headers=headers)